# GPU phases on a single Colab A100: DeepSpec draft-surprise pipeline

Runs phases 1b–5 on one A100 runtime. **Requirements:**

- Colab **A100 GPU** runtime (High-RAM comes with it). Pro+ background execution strongly recommended — the long cells run for hours.
- Secrets (🔑 sidebar): `HF_TOKEN` and `OPENROUTER_API_KEY` (~5–15 $ credit).
- The phase-1a export from the quickstart notebook (re-created below if missing).

**How this fits one GPU:** baseline answers are regenerated via **OpenRouter** (no local serving); hidden-state capture and scoring are forward-only passes of the **FP8 checkpoint** (~35 GB — the exact checkpoint the dataset was generated from); the draft itself is tiny.

| step | wall-clock (rough) | disk |
|---|---|---|
| 1b OpenRouter regen (3k samples) | 1–3 h (API-bound) | <1 GB |
| 2 target cache | 2–5 h | ~30–60 GB |
| 3 draft training | 3–8 h | ~5 GB checkpoints |
| 4 scoring (weird + null) | 1–3 h | <1 GB |
| 5 report | minutes | — |

Every long step is **resumable** — if the session dies, rerun the setup cells and the failed cell; finished work is skipped. Mount Drive (optional cell below) to survive full VM resets.

In [ ]:
# 1) Clone/update repos
import os

if not os.path.exists('/content/WeirdChat'):
    !git clone --branch claude/repo-published-weights-u71yew https://github.com/Erikiss/WeirdChat /content/WeirdChat
else:
    !git -C /content/WeirdChat pull
if not os.path.exists('/content/DeepSpec'):
    !git clone --depth 1 https://github.com/Erikiss/DeepSpec /content/DeepSpec

%cd /content/WeirdChat/examples/03_deepspec_draft_surprise
os.environ['DEEPSPEC_ROOT'] = '/content/DeepSpec'
os.environ['WEIRDSPEC_TARGET_MODEL'] = 'Qwen/Qwen3.6-35B-A3B-FP8'  # the dataset checkpoint

In [ ]:
# 2) Tokens from Colab Secrets (fallback: paste)
import os
from getpass import getpass

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    print('tokens loaded from Colab Secrets')
except Exception:
    os.environ.setdefault('HF_TOKEN', getpass('HF token: '))
    os.environ.setdefault('OPENROUTER_API_KEY', getpass('OpenRouter key: '))

In [ ]:
# 3) (Optional but recommended) mount Drive so results survive VM resets
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/weirdspec/data'
    CKPT_NOTE = 'checkpoints stay on the VM; re-zip them below if you need them off-machine'
else:
    DATA_DIR = '/content/data'
import os; os.makedirs(DATA_DIR, exist_ok=True)
print('DATA_DIR =', DATA_DIR)

In [ ]:
# 4) Install deps + apply the DeepSpec patch (idempotent)
%pip install -q -e /content/WeirdChat 'transformers==5.10.2' datasets

import subprocess
patch = '/content/WeirdChat/examples/03_deepspec_draft_surprise/deepspec_qwen36.patch'
r = subprocess.run(['git', '-C', '/content/DeepSpec', 'apply', '--check', patch], capture_output=True)
if r.returncode == 0:
    subprocess.run(['git', '-C', '/content/DeepSpec', 'apply', patch], check=True)
    print('patch applied')
else:
    r2 = subprocess.run(['git', '-C', '/content/DeepSpec', 'apply', '--reverse', '--check', patch], capture_output=True)
    assert r2.returncode == 0, f'patch neither applies nor is applied:\n{r.stderr.decode()}'
    print('patch already applied')

In [ ]:
# 5) Phase 0 on the FP8 checkpoint, with the GPU gate — must end all-green
!python phase0_feasibility.py --deepspec-root $DEEPSPEC_ROOT --gpu --deep

import json, os
report = json.load(open('phase0_report.json'))
assert report['all_hard_gates_passed'], 'gates failed - fix before spending GPU hours'
os.environ['WEIRDSPEC_MASK_TOKEN_ID'] = str(report['recommended_mask_token_id'])
os.environ['WEIRDSPEC_DRAFT_INTERMEDIATE_SIZE'] = str(report['recommended_draft_intermediate_size'])
LAYER_IDS = str(report['recommended_target_layer_ids']).replace(' ', '')
print('env set; layer ids:', LAYER_IDS, '| gpu:', report.get('gpu'))

In [ ]:
# 6) Phase 1a — weird transcripts (skipped if already exported)
import os
if not os.path.exists(f'{DATA_DIR}/weird_transcripts.jsonl'):
    !python phase1_export_weirdchat.py --output-dir {DATA_DIR}
else:
    print('weird export already present')

In [ ]:
# 7) Phase 1b — prompts + OpenRouter regeneration (resumable; ~1-3 h)
import os
if not os.path.exists(f'{DATA_DIR}/perfectblend_train.jsonl'):
    !python $DEEPSPEC_ROOT/scripts/data/download_and_split.py \
        --dataset-name mlabonne/open-perfectblend --test-size 0.05 \
        --train-output-path {DATA_DIR}/perfectblend_train.jsonl \
        --test-output-dir {DATA_DIR}/eval_heldout --skip-existing

!python phase1b_openrouter.py --input {DATA_DIR}/perfectblend_train.jsonl \
    --output-dir {DATA_DIR} --max-samples 3000 --concurrency 16

In [ ]:
# 8) Phase 2 — target cache (~2-5 h; resumable per rank shard)
!cd $DEEPSPEC_ROOT && python scripts/data/prepare_target_cache.py \
    --config /content/WeirdChat/examples/03_deepspec_draft_surprise/dspark_qwen36_35b_a3b_colab.py \
    --train-data-path {DATA_DIR}/baseline_train.jsonl \
    --output-dir /content/cache/qwen36_colab_target_cache \
    --local-batch-size 2 --num-workers 2 \
    --opts model.target_layer_ids="{LAYER_IDS}"

!du -sh /content/cache/qwen36_colab_target_cache

In [ ]:
# 9) Phase 3 — draft training (~3-8 h; auto-resumes from step_latest on rerun)
!cd $DEEPSPEC_ROOT && CUDA_VISIBLE_DEVICES=0 python train.py \
    --config /content/WeirdChat/examples/03_deepspec_draft_surprise/dspark_qwen36_35b_a3b_colab.py \
    --opts data.target_cache_path=/content/cache/qwen36_colab_target_cache \
    --opts model.target_layer_ids="{LAYER_IDS}"

import glob
DRAFT = sorted(glob.glob(os.path.expanduser('~/checkpoints/weirdspec/dspark_block7_qwen36_35b_a3b_colab/step_*')))[-1]
print('draft checkpoint:', DRAFT)

In [ ]:
# 10) Phase 4 — surprise scoring: weird set, then baseline null (~1-3 h)
!python phase4_score_surprise.py --deepspec-root $DEEPSPEC_ROOT \
    --draft {DRAFT} --target $WEIRDSPEC_TARGET_MODEL \
    --data {DATA_DIR}/weird_transcripts.jsonl --output {DATA_DIR}/weird_scores.jsonl \
    --max-length 2048
!python phase4_score_surprise.py --deepspec-root $DEEPSPEC_ROOT \
    --draft {DRAFT} --target $WEIRDSPEC_TARGET_MODEL \
    --data {DATA_DIR}/baseline_heldout.jsonl --output {DATA_DIR}/baseline_scores.jsonl \
    --max-length 2048

In [ ]:
# 11) Phase 5 — the interpretation report
!python phase5_analyze.py \
    --weird-scores {DATA_DIR}/weird_scores.jsonl \
    --weird-meta {DATA_DIR}/weird_meta.jsonl \
    --baseline-scores {DATA_DIR}/baseline_scores.jsonl \
    --output {DATA_DIR}/report.md

from IPython.display import Markdown, display
display(Markdown(open(f'{DATA_DIR}/report.md').read()))

In [ ]:
# 12) Download everything that matters
!zip -qj /content/weirdspec_results.zip {DATA_DIR}/report.md \
    {DATA_DIR}/weird_scores.jsonl {DATA_DIR}/baseline_scores.jsonl \
    {DATA_DIR}/weird_meta.jsonl phase0_report.json
from google.colab import files
files.download('/content/weirdspec_results.zip')

## Notes

- **Session died mid-step?** Rerun cells 1–5 (fast), then only the failed step's cell — regen, cache, and training all resume.
- **OOM in phase 2/4?** Drop `--local-batch-size` to 1 (phase 2) and `--max-length` to 1536.
- **OpenRouter vs. FP8 caveat:** baseline texts come from whatever quantization OpenRouter's provider serves, while hidden states come from the FP8 checkpoint — the same replication gap WeirdChat documents. Fine for this scale; a rented multi-GPU node serving FP8 locally closes it.
- Scale knobs: `--max-samples` in cell 7 (corpus), `num_train_epochs` via `--opts train.num_train_epochs=...` in cell 9.